# Py-HLA-Match · Demo Benchmark Notebook  
For research use only | Fraunhofer-Gesellschaft zur Förderung der angewandten Forschung e.V. 

⚠️ Disclaimer  

Py-HLA-Match is not certified or conformity assessed as a medical device software or in-vitro medical device software and is intended for research use only. It must therefore not be used for diagnosis or therapy of patients.

For more details on intended use, scope, and limitations, see the [Software Card](https://github.com/fraunhofer-izi/py-hla-match/blob/main/SOFTWARE_CARD.md).

In [1]:
import pandas as pd

from scripts.performance_benchmark import benchmark_hla_match

In [2]:
# NOTE: this import executes the data generator
%run scripts/synthetic_data_generator.py

Successfully generated 'data/synthetic_data/recipients_1000.csv' and 'data/synthetic_data/donors_1000.csv'.
Total Records: 1000 per file.
Guaranteed Matches: 900 (90%).
Successfully generated 'data/synthetic_data/recipients_10000.csv' and 'data/synthetic_data/donors_10000.csv'.
Total Records: 10000 per file.
Guaranteed Matches: 9000 (90%).
Successfully generated 'data/synthetic_data/recipients_100000.csv' and 'data/synthetic_data/donors_100000.csv'.
Total Records: 100000 per file.
Guaranteed Matches: 90000 (90%).
Successfully generated 'data/synthetic_data/recipients_200000.csv' and 'data/synthetic_data/donors_200000.csv'.
Total Records: 200000 per file.
Guaranteed Matches: 180000 (90%).
Successfully generated 'data/synthetic_data/recipients_500000.csv' and 'data/synthetic_data/donors_500000.csv'.
Total Records: 500000 per file.
Guaranteed Matches: 450000 (90%).
Successfully generated 'data/synthetic_data/recipients_1000000.csv' and 'data/synthetic_data/donors_1000000.csv'.
Total Recor

In [4]:
dataset_sizes = [1000, 10000, 100000, 200000, 500000, 1000000]

results = []
for n in dataset_sizes:
    patient_path = f"data/synthetic_data/recipients_{n}.csv"
    donor_path   = f"data/synthetic_data/donors_{n}.csv"
    output_path  = f"data/synthetic_data/match_results_{n}.csv"
    
    stats = benchmark_hla_match(
        donor_path,
        patient_path,
        output_path,
        n_patients=n,
        n_donors=n,
        chunk_size=100000,  # the bigger the better, because results are cached
        verbose=False
    )
    results.append(stats)

Creating /tmp/pyard-georg/pyard-Latest.sqlite3 as cache.
Version: 3640


In [5]:
df = pd.DataFrame(results)[
    ["patients", "donors", "time_s", "comparisons_per_s", "mem_peak_mb"]
]

df.rename(columns={
    "patients": "patients",
    "donors": "donors",
    "time_s": "time [s]",
    "comparisons_per_s": "throughput [/s]",
    "mem_peak_mb": "mem peak [MB]"
}, inplace=True)

df

,patients,donors,time [s],throughput [/s],mem peak [MB]
0,1000,1000,23.11,43,2421.0
1,10000,10000,9.23,1083,2427.6
2,100000,100000,92.34,1082,2419.3
3,200000,200000,184.06,1086,2442.7
4,500000,500000,455.68,1097,2423.3
5,1000000,1000000,916.59,1091,2423.3


In [3]:
import platform, psutil, subprocess, sys
import py_hla_match

print("=" * 50)
print("VM SPECIFICATIONS")
print("=" * 50)

specs = {
    "Python":          sys.version.split()[0],
    "Platform":        platform.platform(),
    "Architecture":    platform.machine(),
    "Physical cores":  psutil.cpu_count(logical=False),
    "Logical cores":   psutil.cpu_count(logical=True),
    "RAM (GB)":        round(psutil.virtual_memory().total / (1024**3), 1),
    "py-hla-match":    py_hla_match.__version__,
}

# CPU model (Linux)
try:
    out = subprocess.check_output(["lscpu"], text=True, stderr=subprocess.DEVNULL)
    for line in out.splitlines():
        if "Model name" in line:
            specs["CPU model"] = line.split(":", 1)[1].strip()
            break
except Exception:
    pass

# Disk
try:
    import shutil
    total, used, free = shutil.disk_usage("/")
    specs["Disk total (GB)"] = round(total / (1024**3), 1)
    specs["Disk free (GB)"]  = round(free / (1024**3), 1)
except Exception:
    pass

for k, v in specs.items():
    print(f"  {k:20s}: {v}")

VM SPECIFICATIONS
  Python              : 3.12.3
  Platform            : Linux-6.8.0-110-generic-x86_64-with-glibc2.39
  Architecture        : x86_64
  Physical cores      : 20
  Logical cores       : 20
  RAM (GB)            : 62.9
  py-hla-match        : 0.1.0
  CPU model           : Intel(R) Xeon(R) Silver 4210 CPU @ 2.20GHz
  Disk total (GB)     : 491.6
  Disk free (GB)      : 36.2
